In [9]:
import json
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [1]:
def add_number( num1, num2):
    print( 'call', num1, num2)
    return {'합':num1+num2}

In [2]:
add_number(5, 3)

call 5 3


{'합': 8}

In [3]:
# OpenAI API 모델에 전달할 Tool(함수 호출) 정의 리스트입니다.
sum_tool = [{
    # 이 도구가 실행 가능한 함수(function) 형태임을 명시합니다.
    'type': 'function',
    'function': {
        # GPT가 호출할 파이썬 함수 이름을 지정합니다.
        'name': 'add_number',
        # 모델이 어떤 요청에서 이 함수를 사용할지 판단하는 설명입니다.
        'description': '두 숫자의 합을 계산합니다.',
        # 함수 실행 시 전달할 매개변수(인자)의 구조를 JSON Schema 형태로 정의합니다.
        'parameters': {
            # 매개변수 집합의 기본 데이터 타입을 객체(Object/Dict)로 지정합니다.
            'type': 'object',
            # 함수에 전달될 각 인자의 속성을 정의합니다.
            'properties': {
                # 첫 번째 덧셈 대상 숫자 속성이며, 데이터 타입은 숫자(number)입니다.
                'num1': {'type': 'number'},
                # 두 번째 덧셈 대상 숫자 속성이며, 데이터 타입은 숫자(number)입니다.
                'num2': {'type': 'number'}
            },
            # 함수 호출 시 반드시 전달되어야 하는 필수 인자 목록입니다.
            'required': ['num1', 'num2']
        }
    }
}]



In [10]:

messages=[{'role':'user','content':'5와 3을 더해 주세요'}]

response = client.chat.completions.create(
    model='gpt-4o-mini',
    messages= messages,
     tools=sum_tool )

print( response.choices[0].message.tool_calls[0].function.arguments )
args = json.loads( response.choices[0].message.tool_calls[0].function.arguments )
result = add_number( args['num1'], args['num2'] )
# result {'합':8}
messages.append( {'role':'function','name':'add_number','content':json.dumps(result)} )
# messages=[{'role':'user','content':'5와 3을 더해 주세요'},
# {'role':'function','name':'add_number','content':"{'합':8}" } ]
response =client.chat.completions.create( model='gpt-3.5-turbo',messages= messages )
print( response.choices[0].message.content)

{"num1":5,"num2":3}
call 5 3
5와 3을 더하면 8이 됩니다.


In [11]:


messages=[{'role':'user','content':'5와 3을 더해 주세요'}]

response = client.chat.completions.create(
    model='gpt-4o-mini',
    messages= messages,
     tools=sum_tool )

print( response.choices[0].message.tool_calls[0].function.arguments )
args = json.loads( response.choices[0].message.tool_calls[0].function.arguments )
result = add_number( args['num1'], args['num2'] )
# result {'합':8}
messages.append( {'role':'function','name':'add_number','content':json.dumps(result)} )
# messages=[{'role':'user','content':'5와 3을 더해 주세요'},
# {'role':'function','name':'add_number','content':"{'합':8}" } ]
response =client.chat.completions.create( model='gpt-3.5-turbo',messages= messages )
print( response.choices[0].message.content)

{"num1":5,"num2":3}
call 5 3
5와 3을 더하면 8입니다.


In [12]:
4. 시간 설정
- 6대륙
  - 아시아 (Asia) : 중국, 한국, 일본, 인도, 러시아(아시아 지역) 등
  - 유럽 (Europe) : 영국, 독일, 프랑스, 이탈리아, 러시아(유럽 지역) 등
  - 아프리카 (Africa) : 북아프리카(이집트, 모로코)부터 사하라 이남 아프리카까지 포함
  - 아메리카 (America) : 미국, 캐나다, 브라질, 아르헨티나 등
  - 오세아니아 (Oceania) : 호주, 뉴질랜드, 파푸아뉴기니, 태평양 도서국 포함
  - 남극 (Antarctica)

SyntaxError: invalid syntax (3147956590.py, line 1)

In [13]:
import pytz
from datetime import datetime
# tz = pytz.timezone( 'Asia/Seoul')
tz = pytz.timezone( 'America/New_York')
local_time = datetime.now( tz )
local_time.strftime( '시간:%Y-%m-%d %H:%M:%S')

'시간:2026-09-10 03:43:50'

In [14]:
def get_time( city ):
    timezones={
        '서울':'Asia/Seoul',
        '도쿄':'Asia/Tokyo',
        '뉴욕':'America/New_York',
        '런던':'Europe/London'
    }
    try:
        tz = pytz.timezone( timezones[city] )
        local_time = datetime.now( tz )
        return local_time.strftime( '시간:%Y-%m-%d %H:%M:%S')
    except Exception as err:
        return "현재 정보로는 시간을 알수 없습니다."

In [15]:
get_time('서울')

'시간:2026-09-10 16:44:04'

In [16]:
get_time('뉴욕')

'시간:2026-09-10 03:44:10'

5. wikipedia

| 함수                                      | 설명                                      |
| --------------------------------------- | --------------------------------------- |
| `wikipedia.search(query)`               | 키워드 검색 결과(문서 제목 리스트) 반환                 |
| `wikipedia.summary(title, sentences=n)` | 해당 문서의 요약을 n문장 반환                       |
| `wikipedia.page(title)`                 | 문서 전체 페이지 객체 반환 (title, url, content 등) |
| `wikipedia.set_lang("ko")`              | 검색 언어 설정 (예: 한국어 `"ko"`, 영어 `"en"`)     |


In [18]:
import wikipedia
from wikipedia.exceptions import DisambiguationError, PageError


def get_wikipedia_info(query: str, lang: str = "ko", sentences: int = 3):
    """위키피디아에서 검색어를 기반으로 요약 및 상세 페이지 정보를 안전하게 조회합니다.

    :param query: 검색할 키워드
    :param lang: 위키피디아 언어 설정 ('ko', 'en' 등)
    :param sentences: 요약문 문장 개수
    """
    # 1. 언어 설정
    wikipedia.set_lang(lang)

    print(f"=== [{query}] 위키피디아 검색 시작 (언어: {lang}) ===")

    # 2. 관련 연관 검색어 목록 가져오기
    search_results = wikipedia.search(query)
    print(f"1. 연관 검색어 목록: {search_results}\n")

    try:
        # 3. 문서 요약 가져오기
        summary_text = wikipedia.summary(query, sentences=sentences)
        print(f"2. 요약 정보 ({sentences}문장):\n{summary_text}\n")

        # 4. 전체 페이지 객체 가져오기
        page = wikipedia.page(query)

        print("3. 상세 페이지 정보:")
        print(f"   - 문서 제목: {page.title}")
        print(f"   - 문서 URL : {page.url}")
        print(f"   - 본문 요약 (앞 300자):\n{page.content[:300]}...\n")

        return {
            "title": page.title,
            "url": page.url,
            "summary": summary_text,
            "content_preview": page.content[:300],
        }

    except DisambiguationError as e:
        print(
            f"[경고] '{query}' 항목은 동음이의어 문서입니다. 아래 옵션 중 하나로 재검색하세요:"
        )
        print(f"   추천 검색어 목록: {e.options[:5]}")
        return None

    except PageError:
        print(f"[오류] '{query}'에 해당하는 위키피디아 페이지를 찾을 수 없습니다.")
        return None

    except Exception as e:
        print(f"[오류] 처리 중 알 수 없는 에러가 발생했습니다: {str(e)}")
        return None


# ----------------------------------------------------
# 실행 테스트
# ----------------------------------------------------
if __name__ == "__main__":
    # 테스트 1: 영어 검색 (Python 프로그래밍 언어)
    get_wikipedia_info("Python (programming language)", lang="en", sentences=2)

    print("-" * 60)

    # 테스트 2: 한국어 검색 (인공지능)
    get_wikipedia_info("인공지능", lang="ko", sentences=3)

=== [Python (programming language)] 위키피디아 검색 시작 (언어: en) ===


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [19]:
import wikipedia

def wikipedia_search(question):
    wikipedia.set_lang("ko")
    try:
        search_result = wikipedia.search(question)[0]
        print('search_result:', search_result)
        wiki_summary = wikipedia.summary(search_result, sentences=5)
    except Exception as e:
        wiki_summary = "위키피디아에서 정보를 찾을 수 없습니다."
    return {"summary": wiki_summary}

wikipedia_search("세종대왕은 누구야?" )

search_result: 대왕 세종


{'summary': '《대왕 세종(大王世宗)》은 2008년 1월 5일에서부터 2008년 11월 16일까지 방송되었던 KBS 2TV 대하 드라마 작품이다.\n\n\n== 트리비아 ==\n조선 태종 시대부터 세종 시대를 대부분의 배경으로 하고 있다. 제1회 방송 시작 하루 전인 2008년 1월 4일에 제작 뒷이야기를 다룬 스페셜 방송이 있었으며, 2008년 11월 16일까지 총 86부작으로 방송되었다.\n당초 KBS 드라마 PD인 이성주가 담당 PD로 낙점되었으나, KBS 드라마 2팀장직으로 발령되면서 연출자가 바뀌었다.\n아울러 2007년 12월 첫 회가 나갈 예정이었지만 전작 《대조영》의 연장에 따라 2008년 1월로 첫 방송일이 변경되었다.'}

In [20]:
import json
import wikipedia
from openai import OpenAI


def wikipedia_search(question):
    wikipedia.set_lang("ko")
    try:
        search_result = wikipedia.search(question)[0]
        wiki_summary = wikipedia.summary(search_result, sentences=5)
    except Exception as e:
        wiki_summary = "위키피디아에서 정보를 찾을 수 없습니다."
    return {"summary": wiki_summary}


wikifunc = [{
    "type": "function",
    "function": {
        "name": "wikipedia_search",
        "description": "입력된 질문에 대해 필요하다면 위키피디아에서  정보를 검색합니다.",
        "parameters": {
            "type": "object",
            "properties": {
                "question": {
                    "type": "string",
                    "description": " 주제 또는 질문"
                }
            },
            "required": ["question"]
        }
    }
}]


messages = [{"role": "user", "content": "세종대왕에 대해 알려줘"}]

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    tools=wikifunc,
    tool_choice="auto"
)


tool_call = response.choices[0].message.tool_calls[0]
print(tool_call)
args = json.loads(tool_call.function.arguments)
print( args)
result = wikipedia_search(args["question"])
print(result)

# messages = [{"role": "user", "content": "세종대왕에 대해 알려줘"},
#             {"role": "function","name": "wikipedia_search",
#              "content":'{summany:위키피디아답변}'}]

messages.append({
    "role": "function",
    "name": "wikipedia_search",
    "content": json.dumps(result)
})

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    temperature=0
)


print('--------------')
print(response.choices[0].message.content)


ChatCompletionMessageFunctionToolCall(id='call_1mwe0kCBCdCIQG8oQwboBUHi', function=Function(arguments='{"question":"세종대왕"}', name='wikipedia_search'), type='function')
{'question': '세종대왕'}
{'summary': '위키피디아에서 정보를 찾을 수 없습니다.'}
--------------
세종대왕(세종대왕, 1397년 5월 15일 ~ 1450년 4월 8일)은 조선의 제4대 왕으로, 한글을 창제한 것으로 가장 잘 알려져 있습니다. 그는 1418년부터 1450년까지 통치하였으며, 그의 통치 기간 동안 과학, 문화, 예술 등 여러 분야에서 많은 발전이 있었습니다.

세종대왕은 백성을 사랑하고, 그들의 삶을 개선하기 위해 다양한 정책을 시행했습니다. 특히, 농업과 과학 기술의 발전을 위해 노력하였고, 천문학과 의학 등 여러 학문을 장려했습니다. 그의 가장 큰 업적 중 하나는 1443년에 한글을 창제하여 1446년에 반포한 것입니다. 이는 한국어를 보다 쉽게 읽고 쓸 수 있도록 하여, 국민의 교육 수준을 높이는 데 기여했습니다.

세종대왕은 또한 '집현전'이라는 학문 연구 기관을 설립하여 학자들과 함께 다양한 연구를 진행하였고, 이를 통해 조선의 문화와 학문이 크게 발전할 수 있었습니다. 그의 통치 아래에서 조선은 정치적 안정과 문화적 번영을 이루었습니다. 세종대왕은 한국 역사에서 가장 존경받는 왕 중 한 명으로, 그의 업적은 오늘날에도 많은 사람들에게 기억되고 있습니다.
